# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"License: {metadata.license}")
print(f"Version: {metadata.version}")
print(f"Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Display all record sets (`@id`, name, and included fields)
record_sets = list(dataset.record_sets)
print(f"Number of record sets found: {len(record_sets)}\n")
for record_set in record_sets:
    print(f"Record set @id: {record_set.id}")
    print(f"  Name: {record_set.name}")
    print(f"  Description: {record_set.description if hasattr(record_set, 'description') else ''}")
    print("  Fields and columns:")
    for field in record_set.fields:
        field_info = f"    {field.id}: {field.name} (dataType={getattr(field, 'dataType', None)})"
        if hasattr(field, 'column'):
            field_info += f" | column @id: {field.column.id}"
        print(field_info)
    print()
    # Print only first 2 record sets for brevity
    if record_sets.index(record_set) >= 1:
        break

**Note:** For this demonstration, we'll use the main survey record set `@id` from the above listing for data extraction and analysis. (If more than one is present, you can extend analysis analogously for others.)

## 3. Data Extraction
Load data from the selected record set into a DataFrame for analysis. We refer to entities by their `@id` fields as required.

In [ ]:
# --- Manually grab a main record set `@id` from above overview ---
main_record_set_id = None
for record_set in record_sets:
    if 'survey' in record_set.name.lower():
        main_record_set_id = record_set.id
        break
if not main_record_set_id:
    main_record_set_id = record_sets[0].id  # Fallback to first one.

print(f"Using record set @id: {main_record_set_id}")

# Create list of all record set @ids for further extraction
record_set_ids = [rset.id for rset in record_sets]

dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

df = dataframes[main_record_set_id]
print("Columns in DataFrame:", df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations include removing outliers, transforming data distributions, and grouping data by key attributes.

Let's choose a validated numeric field for exploration (e.g., GAD-7 score, PHQ-9 score, or PSQ score). Find the corresponding column and its `@id` from the record set listing.

In [ ]:
# Try to find common survey instrument fields for analysis
gad7_field_id = None
phq9_field_id = None
psq_field_id = None
group_field_id = None

survey_fields = [field for field in dataset.get_record_set(main_record_set_id).fields]
for field in survey_fields:
    fname = field.name.lower()
    if gad7_field_id is None and ('gad' in fname or 'gad-7' in fname):
        gad7_field_id = field.id
    elif phq9_field_id is None and ('phq' in fname or 'phq-9' in fname):
        phq9_field_id = field.id
    elif psq_field_id is None and ('psq' in fname):
        psq_field_id = field.id
    if group_field_id is None and ('gender' in fname or 'sex' in fname):
        group_field_id = field.id

# Use the first one we can
score_field_id = gad7_field_id or phq9_field_id or psq_field_id
if score_field_id is None:
    # Fallback to first float/int field
    for field in survey_fields:
        if getattr(field, 'dataType', '').lower() in ['float', 'integer', 'number']:
            score_field_id = field.id
            break

print(f"Score field for EDA: {score_field_id}")

# Filter for non-missing, reasonable values
df[score_field_id] = pd.to_numeric(df[score_field_id], errors='coerce')
eda_df = df[df[score_field_id].notnull() & (df[score_field_id] > 0)]

threshold = 10
filtered_df = eda_df[eda_df[score_field_id] > threshold]
print(f"Filtered records with {score_field_id} > {threshold}:")
print(filtered_df[[score_field_id]].head())

# Normalize the score
filtered_df[f"{score_field_id}_normalized"] = (filtered_df[score_field_id] - filtered_df[score_field_id].mean()) / filtered_df[score_field_id].std()
print(f"Normalized {score_field_id} for filtered records:")
print(filtered_df[[score_field_id, f"{score_field_id}_normalized"]].head())

# If group field exists, group by it
if group_field_id and group_field_id in df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[score_field_id].mean().to_frame()
    grouped_df.columns = [f"mean_{score_field_id}"]
    print(f"Grouped data by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Histogram of the selected score field
plt.figure(figsize=(8, 5))
sns.histplot(eda_df[score_field_id].dropna(), bins=15, kde=True)
plt.title(f"Distribution of {score_field_id}")
plt.xlabel(score_field_id)
plt.ylabel("Count")
plt.show()

# If group field exists, visualize mean by group
if group_field_id and group_field_id in df.columns:
    group_means = eda_df.groupby(group_field_id)[score_field_id].mean().reset_index()
    plt.figure(figsize=(7, 4))
    sns.barplot(x=group_field_id, y=score_field_id, data=group_means, palette="viridis")
    plt.title(f"Mean {score_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {score_field_id}")
    plt.show()

## 6. Conclusion
In this notebook, we:

- Loaded the dataset's metadata and records using the `mlcroissant` library, referencing all entities by their `@id`.
- Explored the available record sets and fields programmatically.
- Extracted the tabular data for the main survey record set, loaded it into a pandas DataFrame, and performed basic exploratory data analysis (EDA).
- Filtered, normalized, and grouped on key numeric fields such as one of the GAD-7, PHQ-9, or PSQ scores.
- Visualized the score distribution and compared scores by participant attribute (such as gender).

For in-depth analysis, continue by referencing the exact `@id`s for all fields and adapting the code to your analytic goals.

The use of standardized Croissant `@id` references ensures your exploration remains reproducible and dataset-agnostic for AI-ready, FAIR-compliant workflows.

*Feel free to extend this notebook to deeper analyses or visualizations as required for your domain!*